<a href="https://colab.research.google.com/github/deniskropp/kolors-cli/blob/cli/colab-testing2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Make sure you have git-lfs installed (https://git-lfs.com)
#!git lfs install

#!git clone https://huggingface.co/spaces/Kwai-Kolors/Kolors
!git clone https://github.com/deniskropp/kolors-cli

fatal: destination path 'kolors-cli' already exists and is not an empty directory.


In [9]:
!pip install /content/kolors-cli

Processing ./kolors-cli
  Preparing metadata (setup.py) ... done
  Created wheel for kolors-cli: filename=kolors_cli-0.1.0-py3-none-any.whl size=55738 sha256=999e1c956c0131a80074630901a0f16c29d83e825397e16f5ea8f97ac1a082f7
  Stored in directory: /root/.cache/pip/wheels/f0/dd/1b/019eaed170c9c73cb728804accd135dac6a0d1738f8a11dee4
Successfully built kolors-cli
  Attempting uninstall: kolors-cli
    Found existing installation: kolors-cli 0.1.0
    Uninstalling kolors-cli-0.1.0:
      Successfully uninstalled kolors-cli-0.1.0


In [10]:
!pip install --upgrade transformers torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
^C


In [ ]:
from kolors_cli.kolors_cli import run

run()

In [1]:
import argparse
import torch
import random
import numpy as np
from huggingface_hub import snapshot_download
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
from kolors.pipelines import pipeline_stable_diffusion_xl_chatglm_256_ipadapter, pipeline_stable_diffusion_xl_chatglm_256
from kolors.models.modeling_chatglm import ChatGLMModel
from kolors.models.tokenization_chatglm import ChatGLMTokenizer
from kolors.models import unet_2d_condition
from diffusers import AutoencoderKL, EulerDiscreteScheduler, UNet2DConditionModel
from PIL import Image
import torch.quantization

# Determine device and set torch parameters
device = "cuda" if torch.cuda.is_available() else "cpu"  # Prefer CUDA if available
torch.set_num_threads(4)  # Limit CPU threads

# Download model checkpoints (do this once)
ckpt_dir = snapshot_download(repo_id="Kwai-Kolors/Kolors")
ckpt_IPA_dir = snapshot_download(repo_id="Kwai-Kolors/Kolors-IP-Adapter-Plus")

# --- PRELOAD SHARED RESOURCES (keep lightweight) ---
tokenizer = ChatGLMTokenizer.from_pretrained(f'{ckpt_dir}/text_encoder')
scheduler = EulerDiscreteScheduler.from_pretrained(f"{ckpt_dir}/scheduler")
# --- END PRELOAD ---

MAX_SEED = np.iinfo(np.int32).max
MAX_IMAGE_SIZE = 1024

def calibrate_model(model, calibration_dataloader, device):
    """Calibrates the model for quantization."""
    model.eval()
    with torch.no_grad():
        for input_data in calibration_dataloader:
            if isinstance(input_data, Image.Image):
                # Handle PIL Image input
                input_data = torch.from_numpy(np.array(input_data)).unsqueeze(0).float() / 255.0
                if input_data.shape[1] == 3:
                    input_data = input_data.permute(0, 3, 1, 2)  # Change to (N, C, H, W)

                input_data = input_data.to(device) # Move calibration data to the correct device

            elif isinstance(input_data, torch.Tensor):
                 input_data = input_data.to(device)
            model(input_data)

def load_t2i_pipeline(device, ckpt_dir, calibrate=False, calibration_images=None):
    """Loads the text-to-image pipeline with optional quantization."""
    text_encoder = ChatGLMModel.from_pretrained(f'{ckpt_dir}/text_encoder', torch_dtype=torch.float16).half().to(device)
    vae = AutoencoderKL.from_pretrained(f"{ckpt_dir}/vae", revision=None).half().to(device)
    unet_t2i = UNet2DConditionModel.from_pretrained(f"{ckpt_dir}/unet", revision=None).half().to(device)

    # --- Quantization ---
    if calibrate:
        unet_t2i.eval()
        text_encoder.eval()

        qconfig = torch.quantization.get_default_qconfig('x86')  # Use 'qnnpack' for ARM CPUs
        unet_t2i.qconfig = qconfig
        text_encoder.qconfig = qconfig

        torch.quantization.prepare(unet_t2i, inplace=True)
        torch.quantization.prepare(text_encoder, inplace=True)
        #Prepare a calibration data loader.
        calibration_dataloader = []

        if calibration_images:
            for image_path in calibration_images:
                try:
                    img = Image.open(image_path).convert("RGB")  # Ensure image is in RGB format
                    calibration_dataloader.append(img)
                except Exception as e:
                    print(f"Error loading calibration image {image_path}: {e}")


        calibrate_model(unet_t2i, calibration_dataloader, device)
        calibrate_model(text_encoder, calibration_dataloader, device)  # You might need a different calibration method for text_encoder
        torch.quantization.convert(unet_t2i, inplace=True)
        torch.quantization.convert(text_encoder, inplace=True)
    # --- End Quantization ---

    pipe_t2i = pipeline_stable_diffusion_xl_chatglm_256.StableDiffusionXLPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet_t2i,
        scheduler=scheduler,
        force_zeros_for_empty_prompt=False
    ).to(device)
    return pipe_t2i

def load_i2i_pipeline(device, ckpt_dir, ckpt_IPA_dir, calibrate=False, calibration_images=None):
    """Loads the image-to-image pipeline with optional quantization."""
    text_encoder = ChatGLMModel.from_pretrained(f'{ckpt_dir}/text_encoder', torch_dtype=torch.float16).half().to(device)
    vae = AutoencoderKL.from_pretrained(f"{ckpt_dir}/vae", revision=None).half().to(device)
    unet_i2i = unet_2d_condition.UNet2DConditionModel.from_pretrained(f"{ckpt_dir}/unet", revision=None).half().to(device)
    image_encoder = CLIPVisionModelWithProjection.from_pretrained(f'{ckpt_IPA_dir}/image_encoder', ignore_mismatched_sizes=True).to(dtype=torch.float16, device=device)
    ip_img_size = 336
    clip_image_processor = CLIPImageProcessor(size=ip_img_size, crop_size=ip_img_size)

    # --- Quantization ---
    if calibrate:
        unet_i2i.eval()
        text_encoder.eval()
        image_encoder.eval()

        qconfig = torch.quantization.get_default_qconfig('x86')  # Use 'qnnpack' for ARM CPUs
        unet_i2i.qconfig = qconfig
        text_encoder.qconfig = qconfig
        image_encoder.qconfig = qconfig
        #Prepare a calibration data loader.
        calibration_dataloader = []

        if calibration_images:
            for image_path in calibration_images:
                try:
                    img = Image.open(image_path).convert("RGB")
                    calibration_dataloader.append(img)
                except Exception as e:
                    print(f"Error loading calibration image {image_path}: {e}")

        torch.quantization.prepare(unet_i2i, inplace=True)
        torch.quantization.prepare(text_encoder, inplace=True)
        torch.quantization.prepare(image_encoder, inplace=True)

        calibrate_model(unet_i2i, calibration_dataloader, device)
        calibrate_model(text_encoder, calibration_dataloader, device)
        calibrate_model(image_encoder, calibration_dataloader, device)
        torch.quantization.convert(unet_i2i, inplace=True)
        torch.quantization.convert(text_encoder, inplace=True)
        torch.quantization.convert(image_encoder, inplace=True)
    # --- End Quantization ---

    pipe_i2i = pipeline_stable_diffusion_xl_chatglm_256_ipadapter.StableDiffusionXLPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet_i2i,
        scheduler=scheduler,
        image_encoder=image_encoder,
        feature_extractor=clip_image_processor,
        force_zeros_for_empty_prompt=False
    ).to(device)

    if hasattr(pipe_i2i.unet, 'encoder_hid_proj'):
        pipe_i2i.unet.text_encoder_hid_proj = pipe_i2i.unet.encoder_hid_proj
    pipe_i2i.load_ip_adapter(f'{ckpt_IPA_dir}', subfolder="", weight_name=["ip_adapter_plus_general.bin"])
    return pipe_i2i
def unload_pipeline(pipe):
    """Unloads the pipeline and clears CUDA cache."""
    if pipe is not None:
        pipe.to("cpu")  # Move to CPU first
        del pipe
    torch.cuda.empty_cache()  # Clear CUDA cache

def infer(prompt,
          ip_adapter_image=None,
          ip_adapter_scale=0.5,
          negative_prompt="",
          seed=0,
          randomize_seed=False,
          width=1024,
          height=1024,
          guidance_scale=5.0,
          num_inference_steps=25,
          calibrate = False,
          calibration_images=None): # Added calibrate flag.
    if randomize_seed:
        seed = random.randint(0, MAX_SEED)
    generator = torch.Generator(device=device).manual_seed(seed)

    if ip_adapter_image is None:
        # Text-to-Image
        pipe_t2i = load_t2i_pipeline(device, ckpt_dir, calibrate, calibration_images) # Pass calibrate flag
        image = pipe_t2i(
            prompt=prompt,
            negative_prompt=negative_prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            width=width,
            height=height,
            generator=generator
        ).images[0]
        unload_pipeline(pipe_t2i)  # Unload immediately after use

    else:
        # Image-to-Image
        pipe_i2i = load_i2i_pipeline(device, ckpt_dir, ckpt_IPA_dir, calibrate, calibration_images)  # Pass calibrate flag
        pipe_i2i.set_ip_adapter_scale([ip_adapter_scale])
        image = pipe_i2i(
            prompt=prompt,
            ip_adapter_image=[ip_adapter_image],
            negative_prompt=negative_prompt,
            height=height,
            width=width,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            num_images_per_prompt=1,
            generator=generator
        ).images[0]
        unload_pipeline(pipe_i2i)  # Unload immediately after use

    return image

def main():
    parser = argparse.ArgumentParser(description="Kolors CLI for generating images based on prompts.")
    parser.add_argument("--prompt", type=str, required=True, help="The text prompt for image generation.")
    parser.add_argument("--ip_adapter_image", type=str, default=None, help="Path to the image prompt (optional).")
    parser.add_argument("--ip_adapter_scale", type=float, default=0.5, help="Image influence scale (default: 0.5).")
    parser.add_argument("--negative_prompt", type=str, default="", help="Negative prompt (default: '').")
    parser.add_argument("--seed", type=int, default=0, help="Seed for random number generation (default: 0).")
    parser.add_argument("--randomize_seed", action="store_true", help="Randomize the seed (default: False).")
    parser.add_argument("--width", type=int, default=1024, help="Width of the generated image (default: 1024).")
    parser.add_argument("--height", type=int, default=1024, help="Height of the generated image (default: 1024).")
    parser.add_argument("--guidance_scale", type=float, default=5.0, help="Guidance scale (default: 5.0).")
    parser.add_argument("--num_inference_steps", type=int, default=25, help="Number of inference steps (default: 25).")
    parser.add_argument("--output", type=str, default="output.png", help="Output file path (default: output.png).")
    parser.add_argument("--calibrate", action="store_true", help="Enable calibration for quantization.")  # New argument
    parser.add_argument("--calibration_images", type=str, nargs='+', help="Paths to calibration images.") # New argument

    args = parser.parse_args()

    if args.ip_adapter_image:
        try:
            ip_adapter_image = Image.open(args.ip_adapter_image)
        except Exception as e:
            print(f"Error opening image file {args.ip_adapter_image}: {e}")
            return
    else:
        ip_adapter_image = None

    try:
        image = infer(
            prompt=args.prompt,
            ip_adapter_image=ip_adapter_image,
            ip_adapter_scale=args.ip_adapter_scale,
            negative_prompt=args.negative_prompt,
            seed=args.seed,
            randomize_seed=args.randomize_seed,
            width=args.width,
            height=args.height,
            guidance_scale=args.guidance_scale,
            num_inference_steps=args.num_inference_steps,
            calibrate=args.calibrate,  # Pass the calibrate flag
            calibration_images=args.calibration_images # Pass calibration images.
        )
        image.save(args.output)
        print(f"Image saved to {args.output}")
    except Exception as e:
        print(f"Error during image generation: {e}")

if __name__ == "__main__":
    main()


RuntimeError: Failed to import transformers.models.clip.image_processing_clip because of the following error (look up to see its traceback):
module 'torch.library' has no attribute 'register_fake'